In [9]:
import wandb

wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

# use google colab, everytime must do this
pip install wandb
import wandb
wandb.login()
**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**

W&B 是一个机器学习实验管理平台。
功能： 它像是一个“实验日志本”，自动帮你记录代码运行时的损失曲线（Loss）、准确率（Accuracy）、模型参数、甚至是你电脑的 GPU 占用率。
云端存储： 它把这些数据传到云端，你可以通过网页随时查看，不用自己手动用 Excel 记实验结果。
没有 W&B，你依然可以用 PyTorch 训练模型，但你很难记得住昨天跑的那个模型效果好是因为哪个参数；有了 W&B，你的整个训练过程变得透明且可追溯。

In [10]:
import wandb
import random

import datetime

# 获取当前时间
now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
wandb.init(
    project="cs336-a5-sft-v2",          # 项目名（必选）
    entity="chwmwei-42",             # 你的 W&B 用户名
    name=f"wanda_sft_{now}",        # 可读 run 名称
    config={
        "model": "Qwen2.5-Math-1.5B",
        "dataset_tag": "raw", # raw, sf, grpo
        "batch_size": 64,
        "max_examples": "1000",
        "seed": 2026,
        "learning_rate": 2e-5,
    }
)

epochs = 10
for epoch in range(epochs):
    # 模拟计算出的损失值
    train_loss = 1.0 / (epoch + 1) + random.random() * 0.1
    val_accuracy = 0.5 + (epoch * 0.04)

    # 2. 关键步骤：使用 wandb.log 记录每一轮的数据
    # 字典中的 key 会变成图表的标题，value 是当前数值
    wandb.log({
        "epoch": epoch,
        "train/loss": train_loss,
        "val/accuracy": val_accuracy
    })

    print(f"Epoch {epoch}: loss={train_loss:.4f}")

wandb.finish()



Epoch 0: loss=1.0831
Epoch 1: loss=0.5553
Epoch 2: loss=0.3374
Epoch 3: loss=0.3260
Epoch 4: loss=0.2121
Epoch 5: loss=0.1816
Epoch 6: loss=0.2076
Epoch 7: loss=0.1508
Epoch 8: loss=0.2082
Epoch 9: loss=0.1557


epoch,▁▂▃▃▄▅▆▆▇█
train/loss,█▄▂▂▁▁▁▁▁▁
val/accuracy,▁▂▃▃▄▅▆▆▇█
epoch,9
train/loss,0.15569
val/accuracy,0.86


# use the model from hugging face
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-Math-1.5B"

# 这一步会自动从 Hugging Face 免费下载模型
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

模型权重 (1.5B 参数)：FP32 需要 ~6GB，FP16 需要 ~3GB。
优化器状态 (AdamW)：需要额外记录每个参数的动量，通常是模型大小的 2-3 倍（~9GB）。
中间激活值：训练时的 forward 过程也会占显存。
结论：不量化的话，14.5GB 的 GPU 跑 SFT 确实非常吃力。
# to replace AdamW
1. 使用 8-bit Adam (最推荐)
这是最简单的“无痛”替换方案。它通过量化优化器状态，将显存占用从每个参数 12 字节降至约 2 字节，且几乎不影响模型精度。
import bitsandbytes as bnb

# 替换原来的 AdamW
optimizer = bnb.optim.Adam8bit(model.parameters(), lr=wandb.config.learning_rate)

2. 使用 SGD（随机梯度下降）
SGD 是最省显存的优化器。它不记录任何动量（如果是基础版），显存消耗几乎为零。
优点：极省显存。
缺点：在训练大模型（LLM）时收敛非常慢，容易陷入局部最优。
# 只有基本权重更新，无额外显存占用
optimizer = torch.optim.SGD(model.parameters(), lr=wandb.config.learning_rate)

3. 使用 Adafactor
Adafactor 是专门为节省显存设计的优化器（常用于 T5 模型）。它通过分解误差矩阵，将显存占用降得非常低，且比 SGD 聪明得多。
from transformers.optimization import Adafactor

optimizer = Adafactor(
    model.parameters(),
    lr=wandb.config.learning_rate,
    scale_parameter=False,
    relative_step=False
)


In [3]:
import os
import torch # Ensure torch is imported at the top
import wandb
import random
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from torch.optim import AdamW
from transformers.optimization import Adafactor # Ensure Adafactor is imported for the optimizer

# Set PYTORCH_ALLOC_CONF to enable expandable segments to avoid fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Set LLM_INT8_THRESHOLD environment variable to prevent NameError in bitsandbytes
os.environ["LLM_INT8_THRESHOLD"] = "6.0"

wandb.init(
    project="cs336",
    name="c_1",
    config={
        "learning_rate": 2e-5,
        "model": "Qwen/Qwen2.5-Math-1.5B",
        "batch_size": 4,
        "epochs":3,
        "max_length": 31
    }
)

model_name = wandb.config.model
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer is None:
    raise RuntimeError(f"Failed to load tokenizer for model: {model_name}. Please check the model name and your network connection.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True # Added to address the ValueError
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto" # 自动分配到 GPU
)

if model is None:
    raise RuntimeError(f"Failed to load model: {model_name}. Please check the model name, available memory, or your network connection.")

# With device_map="auto", the model is already on the appropriate device, so model.to(device) is not needed.
# device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
# model.to(device)
# optimizer = torch.optim.SGD(model.parameters(), lr=wandb.config.learning_rate)

opimizer = Adafactor(
    model.parameters(),
    lr=wandb.config.learning_rate,
    scale_parameter=False,
    relative_step=False
)

def get_random_batch(batch_size, max_length):
    # Ensure input_ids and labels are on the correct device. With device_map="auto",
    # the model inputs should also be on the same device as the model.
    # Using model.device directly to infer the device.
    input_ids = torch.randint(0, tokenizer.vocab_size, (batch_size, max_length)).to(model.device)
    labels = input_ids.clone()
    return {"input_ids": input_ids, "labels": labels}



steps_per_epoch = 20

for epoch in range(wandb.config.epochs):
  model.train()
  for batch_idx in range(steps_per_epoch):
      batch = get_random_batch(wandb.config.batch_size, wandb.config.max_length)
      optimizer.zero_grad()
      outputs = model(**batch)
      loss = outputs.loss
      loss.backward()
      optimizer.step()
      if batch_idx % 5 == 0:
        wandb.log({
            "epoch": epoch,
            "loss": loss.item(),
            "batch_idx": batch_idx,
            "steps_per_epoch": steps_per_epoch,
            "train/global_step": batch_idx + epoch * steps_per_epoch
        })
        print(f"Epoch {epoch} | Step {batch_idx} | Loss: {loss.item():.4f}")



wandb.finish()

KeyboardInterrupt: 

# 初始化
run = wandb.init(config={...})

# 在代码其他地方引用
# 这样保证了你记录的数据和代码实际运行的数据 100% 一致
lr = wandb.config.learning_rate
bs = wandb.config.batch_size

optimizer = AdamW(model.parameters(), lr=lr)


GPU 显存溢出 (OOM) 错误。
简单来说：你的显卡（约 14.56 GB 显存）已经塞满了。Qwen2.5-Math-1.5B 模型虽然只有 15 亿参数，但在加载模型、加载梯度、以及 AdamW 优化器（它会为每个参数存两份动量数据）的共同作用下，显存需求会迅速超过 14GB。
1. 使用 4-bit 量化加载（最推荐）
pip install bitsandbytes accelerate
!pip install bitsandbytes accelerate
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# 配置 4-bit 量化
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# 加载模型时加入 quantization_config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto" # 自动分配到 GPU
)

# 注意：使用了 device_map="auto" 后，不需要再写 model.to(device)
2. 清理显存并重启
3. 使用 float16 精度
至少要使用半精度（FP16）加载，这能直接省下一半显存：
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
).to(device)


In [5]:
import os
import torch
import wandb
import random
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch.optim import AdamW
import datetime

now = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Set PYTORCH_ALLOC_CONF to enable expandable segments to avoid fragmentation
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Set LLM_INT8_THRESHOLD environment variable to prevent NameError in bitsandbytes
os.environ["LLM_INT8_THRESHOLD"] = "6.0"
# 1. 初始化 W&B
wandb.init(
    project="cs336-a5-sft-v2",
    name=f"cs_1_{now}",
    config={
        "learning_rate": 2e-5,
        "model": "Qwen/Qwen2.5-0.5B",
        "batch_size": 4,
        "epochs": 3,
        "max_length": 32
    }
)

# 2. 加载真实模型和分词器
model_name = wandb.config.model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16)

# 将模型移动到 GPU (如果可用)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

#optimizer = AdamW(model.parameters(), lr=wandb.config.learning_rate)
optimizer = torch.optim.SGD(model.parameters(), lr=wandb.config.learning_rate)

# 3. 构造随机测试数据 (模拟训练集)
def get_random_batch(batch_size, max_length):
    # 随机生成一些 token ID，模拟模型输入
    input_ids = torch.randint(0, tokenizer.vocab_size, (batch_size, max_length)).to(device)
    # 语言模型训练中，labels 通常就是 input_ids 本身
    labels = input_ids.clone()
    return {"input_ids": input_ids, "labels": labels}

# 4. 真实的训练循环
steps_per_epoch = 20

for epoch in range(wandb.config.epochs):
    model.train()
    for batch_idx in range(steps_per_epoch):
        # 获取随机数据
        batch = get_random_batch(wandb.config.batch_size, wandb.config.max_length)

        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        # 5. 记录数据到 W&B
        if batch_idx % 5 == 0:
            wandb.log({
                "epoch": epoch,
                "train/loss": loss.item(),
                "train/global_step": batch_idx + epoch * steps_per_epoch
            })
            print(f"Epoch {epoch} | Step {batch_idx} | Loss: {loss.item():.4f}")

# 6. 结束
wandb.finish()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Epoch 0 | Step 0 | Loss: 15.0972
Epoch 0 | Step 5 | Loss: 14.1109
Epoch 0 | Step 10 | Loss: 13.2932
Epoch 0 | Step 15 | Loss: 13.2351
Epoch 1 | Step 0 | Loss: 12.6184
Epoch 1 | Step 5 | Loss: 12.7026
Epoch 1 | Step 10 | Loss: 12.4065
Epoch 1 | Step 15 | Loss: 12.4961
Epoch 2 | Step 0 | Loss: 12.5046
Epoch 2 | Step 5 | Loss: 12.4433
Epoch 2 | Step 10 | Loss: 12.4232
Epoch 2 | Step 15 | Loss: 12.4493


epoch,▁▁▁▁▅▅▅▅████
train/global_step,▁▂▂▃▄▄▅▅▆▇▇█
train/loss,█▅▃▃▂▂▁▁▁▁▁▁
epoch,2
train/global_step,55
train/loss,12.44929
